# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on creating new spark dataframes about the demand in different time series for HVFHV dataset and subsampling for plotting.

Since as a driver, they concentrate on where I go could increase the revenue, so we are focusing on the pickup location in the analysis of demand.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, count
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv_demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"
hvfhv_path = base_dir + '/developed/merged_data/full_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "day_type",
    when(hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]), "Weekday")
    .otherwise("Weekend")
)

hvfhv_sdf.show(5)

In [ ]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hvfhv_sdf.printSchema()

In [ ]:
TOTAL_HOURS = 24
TOTAL_DAYS = 31 + 31 + 30 + 31 + 30 + 31
TOTAL_MONTHS = 6 # the timeline is 6 month
TOTAL_LOCATIONS = hvfhv_sdf.select('PULocationID').distinct().count()

# Hourly Demand:

In [ ]:
# Group by `pickup_hour`, `pickup_date``, `day_type`,
# then count the number of records for each group
hourly_pickup_demand_sdf = hvfhv_sdf.groupBy("pickup_hour", "pickup_date", "day_type") \
                                    .count() \
                                    .withColumnRenamed("count", "hourly_demand")
                                    
# Standardize the hourly revenue
hourly_pickup_demand_sdf = hourly_pickup_demand_sdf.withColumn('mean_hourly_demand',
                                                               col('hourly_demand') / TOTAL_HOURS)

hourly_pickup_demand_sdf = hourly_pickup_demand_sdf.orderBy("pickup_hour")\
                                                   .drop("hourly_demand")

hourly_pickup_demand_sdf.show(5)

In [ ]:
hourly_pickup_demand_sdf.describe().show()

# Hourly Demand Among Day of Week:

In [ ]:
# Group by `day_of_week`, "day_type", "pickup_date", "pickup_hour"
# then count the number of records for each group
hourly_demand_among_day_of_week = hvfhv_sdf.groupBy("pickup_date", "day_of_week",
                                                    "day_type", "pickup_hour") \
                                            .count() \
                                            .orderBy("day_of_week") \
                                            .withColumnRenamed("count",
                                                               "day_by_hour_demand")

# Standardize the hourly revenue
hourly_demand_among_day_of_week = hourly_demand_among_day_of_week.withColumn('day_by_hour_demand',
                                                                             col('day_by_hour_demand') / TOTAL_HOURS)

hourly_demand_among_day_of_week.show(5)

In [ ]:
hourly_demand_among_day_of_week.describe().show()

# Daily Demand:

### Pickup:

In [ ]:
# Group by `pickup_date` and `PULocationID`, then count the number of records for each group
daily_pickup_demand_sdf = hvfhv_sdf.groupBy("pickup_date", "PULocationID") \
                                    .count() \
                                    .orderBy("pickup_date", "PULocationID") \
                                    .withColumnRenamed("count", "daily_demand")

# Standardize the daily revenue
daily_pickup_demand_sdf = daily_pickup_demand_sdf.withColumn('daily_demand', 
                                                             col('daily_demand') / TOTAL_DAYS)

daily_pickup_demand_sdf.show(5)

In [ ]:
daily_pickup_demand_sdf.describe().show()

### Dropoff:

In [ ]:
# Group by `pickup_date` and `PULocationID`, then count the number of records for each group
daily_dropoff_demand_sdf = hvfhv_sdf.groupBy("pickup_date", "DOLocationID") \
                                    .count() \
                                    .orderBy("pickup_date", "DOLocationID") \
                                    .withColumnRenamed("count", "daily_demand")
# Standardize the daily revenue
daily_dropoff_demand_sdf = daily_dropoff_demand_sdf.withColumn('daily_demand', 
                                                               col('daily_demand') / TOTAL_DAYS)

daily_dropoff_demand_sdf.show(5)

In [ ]:
daily_dropoff_demand_sdf.describe().show()

# Monthly Demand:

In [ ]:
# Group by `month`, 
# then count the number of records for each group
monthly_pickup_demand_sdf = hvfhv_sdf.groupBy("month") \
                                     .count() \
                                     .orderBy("month") \
                                     .withColumnRenamed("count", "monthly_demand")

# Standardize the monthly revenue
monthly_pickup_demand_sdf = monthly_pickup_demand_sdf.withColumn('monthly_demand', 
                                                                 col('monthly_demand') / TOTAL_MONTHS)

monthly_pickup_demand_sdf.show(5)

In [ ]:
monthly_pickup_demand_sdf.describe().show()

# Save the Merged Datasets:

Save the hourly pickup demand dataset:

In [ ]:
hour_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_pickup_demand'
hour_path = os.path.join(hour_dir, file_name)
hourly_pickup_demand_sdf.write.mode('overwrite').parquet(hour_path)

Save hourly demand among day of week dataset:

In [ ]:
week_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_among_day_of_week'
week_path = os.path.join(week_dir, file_name)
hourly_demand_among_day_of_week.write.mode('overwrite').parquet(week_path)

Save the daily pickup demand dataset:

In [ ]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_pickup_demand'
day_path = os.path.join(day_dir, file_name)
daily_pickup_demand_sdf.write.mode('overwrite').parquet(day_path)

Save the daily dropoff demand dataset:

In [ ]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_dropoff_demand'
day_path = os.path.join(day_dir, file_name)
daily_dropoff_demand_sdf.write.mode('overwrite').parquet(day_path)

Save the monthly pickup demand dataset:

In [ ]:
month_dir = base_dir + '/developed/merged_data'
file_name = 'monthly_pickup_demand'
month_path = os.path.join(month_dir, file_name)
monthly_pickup_demand_sdf.write.mode('overwrite').parquet(month_path)